In [ ]:
# ===== Language Model Multi-Classification ===== #

In [ ]:
#!pip install protobuf
#!pip install unidic-lite
#!pip install fugashi

In [1]:
# ===== Required Imports ===== #
import torch
from transformers import (
    pipeline,
    AutoTokenizer, 
    AutoModelForPreTraining,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import Dataset
from sklearn.metrics import f1_score, accuracy_score
import pandas as pd
import numpy as np
from jlpt_sentence_parser import (
    detect_n5, detect_n4, detect_n3, detect_n2,  # Individual level functions
    detect_all,          # Detect all levels at once
    detect_by_level,     # Detect specific levels
    get_unique_patterns, # Remove duplicates
    format_matches       # Pretty print results
)

/home/evanxavierhu/Documents/VSCode_Local_Projects/.venv/lib64/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Test the basic regex pattern match
test_df = pd.read_csv("training_sent_labeled.csv")
test_sent_list = test_df["text"]

for i in range(30, 60):
    sentence = test_sent_list[i]
    print(f"Sentence: {sentence}")

    for level, fn in [("N5", detect_n5), ("N4", detect_n4), ("N3", detect_n3), ("N2", detect_n2)]:
        matches = fn(sentence)
        if matches:
            print(f"  [{level}]")
            for grammar_id, grammar_point, meaning in matches:
                print(f"    {grammar_id:4} | {grammar_point:20} | {meaning}")

    print()


Sentence: はい、あと僕たちオンラインレッスンの方をしてるのでもし興味がある方は僕たちと一緒に日本語を勉強しましょう。
  [N5]
      12 | があります                | there is; is (non-living things)
      18 | いっしょに                | together
      32 | ましょう                 | let's ~; shall we ~
      65 | ている                  | ongoing action or current state

Sentence: 今日本冬なんですよ。
  [N5]
      45 | んです                  | to explain something; show emphasis

Sentence: 人間やエルフ、ドワーフといった光の神々の加護を受けた種族、人族。
  [N2]
     163 | といった                 | like; such as~

Sentence: 数は少ないが、ここの能力は人族をはるかに上回る闇に祝福された種族、魔族。

Sentence: 宝玉の魔力を解放するしか手はなさそうだな。

Sentence: [音楽]1人じゃ1000年かけたって無理ね。

Sentence: 全くすっごく寂しかったんだから。
  [N5]
      45 | んです                  | to explain something; show emphasis

Sentence: お、でもカイルちゃん、今日はいつもより大人っぽくて、まるで未来から来たみたい。
  [N5]
      19 | いつも                  | always; usually; habitually
  [N3]
      55 | まるで                  | as if; as though; just like

Sentence: 反省はしてるみたいね。
  [N5]
      65 | ている                  | ongoing action or current s

In [5]:
# Load the training data
training_df = pd.read_csv("training_sent_labeled.csv")
training_df.head(2)

,sentence_id,text,labels
0,0,大きく言うとこの2点です。,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,1,あ、買ったものもダメなの?,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, ..."


In [8]:
import ast
from sklearn.model_selection import train_test_split
from transformers import DefaultDataCollator

# Few-shot multi-label classifier
# 1. Load the model
model_name = "tohoku-nlp/bert-base-japanese-char-v3"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2. Define the labels
N4_grammar_df = pd.read_csv("jlpt_n4_grammars_v1.csv")
labels = N4_grammar_df['grammar_point'].tolist()
num_labels = len(N4_grammar_df)

label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}

# 3. Load model for multi-label classification
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    problem_type="multi_label_classification",
    id2label=id2label,
    label2id=label2id
)

# 4. Prepare data from training_df
# Parse labels if stored as strings (CSV serializes lists as strings)
training_df['labels'] = training_df['labels'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

# Train/val split (80/20)
train_df, val_df = train_test_split(training_df, test_size=0.2, random_state=42)

train_dataset = Dataset.from_pandas(train_df[['text', 'labels']].reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df[['text', 'labels']].reset_index(drop=True))

# 5. Tokenize
def tokenize_function(examples):
    tokenized = tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )
    # Convert labels to float tensors for BCEWithLogitsLoss
    tokenized["labels"] = [
        [float(l) for l in label_list] 
        for label_list in examples["labels"]
    ]
    return tokenized

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)

# 6. Custom data collator — the default collator casts labels to Long,
#    but BCEWithLogitsLoss requires Float for multi-label classification
class MultiLabelCollator(DefaultDataCollator):
    def __call__(self, features, return_tensors=None):
        batch = super().__call__(features, return_tensors=return_tensors)
        batch["labels"] = batch["labels"].float()
        return batch

# 7. Define metrics
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = (torch.sigmoid(torch.tensor(predictions)) > 0.5).int().numpy()
    
    f1_micro = f1_score(labels, predictions, average='micro')
    f1_macro = f1_score(labels, predictions, average='macro')
    accuracy = accuracy_score(labels, predictions)
    
    return {
        "f1_micro": f1_micro,
        "f1_macro": f1_macro,
        "accuracy": accuracy,
    }

# 8. Training arguments (tuned for few-shot)
training_args = TrainingArguments(
    output_dir="./bert-japanese-multilabel",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    
    # Evaluation settings
    eval_strategy="epoch",              # Evaluate after each epoch
    save_strategy="epoch",              # Save model after each epoch
    load_best_model_at_end=True,        # Load best model when finished
    metric_for_best_model="f1_micro",   # Use F1 micro to determine best model
    
    # Logging
    logging_steps=50,
    logging_dir="./logs",
    
    # Report to
    report_to="none",  # Disable wandb/tensorboard if not configured
)

# 9. Train
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=MultiLabelCollator(),
)


print("Starting training with validation...")
print(f"Train samples: {len(train_df)}, Val samples: {len(val_df)}")
print("="*60)

trainer.train()

print("\n" + "="*60)
print("Training complete!")
print("="*60)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 907.63it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: tohoku-nlp/bert-base-japanese-char-v3
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
cl

Starting training with validation...
Train samples: 348, Val samples: 88


/home/evanxavierhu/Documents/VSCode_Local_Projects/.venv/lib64/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib64/python3.13/threading.py", line 1044, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "/usr/lib64/python3.13/threading.py", line 995, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/evanxavierhu/Documents/VSCode_Local_Projects/.venv/lib64/python3.13/site-packages/transformers/safetensors_conversion.py", line 117, in auto_conversion
    raise e
  File "/home/evanxavierhu/Documents/VSCode_Local_Projects/.venv/lib64/python3.13/site-packages/transformers/safetensors_conversion.py", line 96, in auto_conversion
    sha = get_conversion_pr_reference(api, pret

Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,Accuracy
1,0.511502,0.269766,0.000000,0.000000,0.284091
2,0.222977,0.177454,0.000000,0.000000,0.284091
3,0.167440,0.157635,0.000000,0.000000,0.284091


/home/evanxavierhu/Documents/VSCode_Local_Projects/.venv/lib64/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.08it/s]
/home/evanxavierhu/Documents/VSCode_Local_Projects/.venv/lib64/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/home/evanxavierhu/Documents/VSCode_Local_Projects/.venv/lib64/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control t


Training complete!


In [9]:
# 9. Inference - show top 10 predicted labels
def predict(text, top_k=10):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.sigmoid(outputs.logits)
    
    top = torch.topk(probs[0], k=top_k)
    return [(labels[idx], round(score.item(),5)) for score, idx in zip(top.values, top.indices)]

In [11]:
test_df = pd.read_csv("/home/evanxavierhu/Documents/VSCode_Local_Projects/JL_Project/Japanese_Lang_Project/data_exploration/parsed_sentences/【Japanese Podcast】Hay Fever Season Is Here! - Master 701 Essential Words Without Even Noticing.ja.csv")
test_df.head(1)

,sentence_id,start_time,end_time,text,jlpt_level,grammar,vocabulary
0,0,0.0,3.58,はい、みなさん、こんにちは。,NaN,NaN,NaN


In [12]:
# Few-shot multi-label classifier

example_sents = test_df["text"]
example_sent = example_sents[20]
#example_sent = "⾷事の間、彼⼥と⼀緒に映画を⾒ました。"

# # Debug: see the raw probabilities
# inputs = tokenizer(example_sent, return_tensors="pt", truncation=True, max_length=128)
# with torch.no_grad():
#     outputs = model(**inputs)
#     probs = torch.sigmoid(outputs.logits)

# # Show top 10 predictions regardless of threshold
# top_k = torch.topk(probs[0], k=10)
# print(f"Sentence: {example_sent}\n")
# print("Top 10 predictions:")
# for score, idx in zip(top_k.values, top_k.indices):
#     print(f"  {labels[idx]:15s}  {score:.4f}")

#print(f"Max prob: {probs.max():.4f}    Min prob: {probs.min():.4f}")
for i in range(0,50):
    result = predict(example_sents[i])
    print(f"Sentence: {example_sents[i]}")
    print(f"Result: {result}\n")

Sentence: はい、みなさん、こんにちは。
Result: [('たばかり', 0.34254), ('てもらう', 0.32894), ('ように ような', 0.3235), ('かな', 0.30991), ('がする', 0.3054), ('たどうし & じどうし', 0.30329), ('くする', 0.30108), ('でも', 0.29995), ('そうだ', 0.29498), ('きっと', 0.29448)]

Sentence: ジローです。
Result: [('たばかり', 0.34899), ('ように ような', 0.32976), ('てもらう', 0.31893), ('がする', 0.31399), ('かな', 0.30591), ('がる・がっている', 0.30478), ('でも', 0.30262), ('くする', 0.30169), ('きっと', 0.29641), ('ても', 0.29395)]

Sentence: 本日も、僕のポッドキャストを聞きに来てくださり、ありがとうございます。
Result: [('たばかり', 0.33795), ('てもらう', 0.32727), ('ように ような', 0.32491), ('かな', 0.31007), ('たどうし & じどうし', 0.30232), ('くする', 0.30101), ('でも', 0.29792), ('がする', 0.29579), ('きっと', 0.29288), ('そうだ', 0.29219)]

Sentence: 本日も、よろしくお願いします。
Result: [('たばかり', 0.34458), ('てもらう', 0.32877), ('ように ような', 0.32486), ('かな', 0.3087), ('くする', 0.30256), ('がする', 0.30036), ('たどうし & じどうし', 0.29846), ('でも', 0.29814), ('そうだ', 0.29378), ('きっと', 0.29267)]

Sentence: はい、みなさん、お元気でしたか?
Result: [('たばかり', 0.34063), ('てもらう', 0.32544), ('ように ような',

## Understanding Training: Epochs and Weight Adjustment

### What are Epochs?
An **epoch** is one complete pass through the entire training dataset.

- **1 epoch** = The model sees every training example once
- **3 epochs** = The model sees every training example 3 times

### Why Do We Adjust Weights?

Neural networks learn by **adjusting internal parameters (weights)** to minimize errors:

1. **Forward Pass**: Input sentence → Model → Predictions
2. **Loss Calculation**: Compare predictions to true labels → Calculate error
3. **Backward Pass**: Calculate gradients (how to change weights to reduce error)
4. **Weight Update**: Adjust weights in the direction that reduces loss

**Example:**
```
Initial weights: Random → Model predicts "ことができる" with 10% confidence
True label: "間" (should be 100% confidence, 0% for others)
Loss: Very high because prediction is wrong
Gradient: Shows which weights to increase/decrease
Updated weights: Changed slightly to make "間" more likely next time
```

### Why Multiple Epochs?

- **1 epoch** is rarely enough to learn patterns
- Each epoch refines the weights a bit more
- **Too few epochs** = Underfitting (model doesn't learn enough)
- **Too many epochs** = Overfitting (model memorizes training data, performs poorly on new data)

### Training Process (3 epochs example):

```
Epoch 1/3:
  - Process all 841 training sentences
  - Update weights 841 times (or in batches)
  - Check validation accuracy: Maybe 30%

Epoch 2/3:
  - Process same 841 sentences again (with updated weights)
  - Weights are refined further
  - Check validation accuracy: Maybe 50%

Epoch 3/3:
  - Process sentences one more time
  - Weights are nearly optimal
  - Check validation accuracy: Maybe 60%
```

### Key Parameters:

- **Learning rate (2e-5)**: How big each weight adjustment is
  - Too high: Model may overshoot optimal weights
  - Too low: Training takes forever
  
- **Batch size (4)**: How many examples to process before updating weights
  - Larger batches: Faster but less frequent updates
  - Smaller batches: Slower but more frequent updates

### Why We Need Validation Data:

Training data alone can't tell if the model is learning **generalizable patterns** or just **memorizing examples**. 

- **Training loss**: Always decreases (model is memorizing)
- **Validation loss**: Should decrease initially, then plateau or increase if overfitting

By monitoring validation performance, we know when to stop training!

## Single-Label Classification (Alternative Approach)

This version treats the problem as **single-label classification** - predicting the ONE primary grammar pattern in each sentence.

**Key Differences from Multi-Label:**
- **Loss function**: CrossEntropyLoss (instead of BCEWithLogitsLoss)
- **Labels format**: Single integer index (instead of float vector)
- **Output**: Softmax probabilities that sum to 1.0 (instead of independent sigmoids)
- **Prediction**: Argmax to get the single best label (instead of threshold > 0.5 for multiple labels)

**When to use Single-Label:**
- When each sentence has ONE primary grammar pattern
- Simpler task, often performs better with limited data
- Good baseline before attempting multi-label

In [ ]:
# ===== SINGLE-LABEL CLASSIFICATION VERSION ===== #

# 1. Load the model
model_name = "tohoku-nlp/bert-base-japanese-char-v3"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2. Define the labels (same as before)
labels = unique_grammar.tolist()
num_labels = len(unique_grammar)

label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}

# 3. Load model for SINGLE-label classification
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    problem_type="single_label_classification",  # ← Changed from multi_label
    id2label=id2label,
    label2id=label2id
)

# 4. Convert label vectors to single indices
# From: [0.0, 0.0, 1.0, 0.0, ...] → To: 2
def convert_to_single_label(data):
    converted = []
    for item in data:
        # Find the index where label is 1.0
        label_idx = item["labels"].index(1.0)
        converted.append({
            "text": item["text"],
            "label": label_idx  # ← Now a single integer instead of list
        })
    return converted

train_data_single = convert_to_single_label(train_data)
val_data_single = convert_to_single_label(val_data)

print(f"Converted {len(train_data_single)} train samples to single-label format")
print(f"Example: {train_data_single[0]}")

# 5. Prepare datasets
train_dataset = Dataset.from_list(train_data_single)
val_dataset = Dataset.from_list(val_data_single)

# 6. Tokenize (simpler for single-label)
def tokenize_function_single(examples):
    tokenized = tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )
    # Labels are already integers, no conversion needed
    tokenized["label"] = examples["label"]
    return tokenized

tokenized_train = train_dataset.map(tokenize_function_single, batched=True)
tokenized_val = val_dataset.map(tokenize_function_single, batched=True)

# 7. Define metrics (simpler for single-label)
def compute_metrics_single(eval_pred):
    predictions, labels = eval_pred
    # Get the class with highest probability
    predictions = predictions.argmax(axis=1)
    
    # Calculate metrics
    f1_micro = f1_score(labels, predictions, average='micro')
    f1_macro = f1_score(labels, predictions, average='macro')
    f1_weighted = f1_score(labels, predictions, average='weighted')
    accuracy = accuracy_score(labels, predictions)
    
    return {
        "accuracy": accuracy,
        "f1_micro": f1_micro,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted,
    }

# 8. Training arguments (same as before)
training_args = TrainingArguments(
    output_dir="./bert-japanese-singlelabel",  # Different output dir
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    
    # Evaluation settings
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",  # Weighted F1 is better for imbalanced classes
    
    # Logging
    logging_steps=50,
    logging_dir="./logs",
    report_to="none",
)

# 9. Train
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics_single,
)

print("\nStarting SINGLE-LABEL training...")
print(f"Train samples: {len(train_data_single)}, Val samples: {len(val_data_single)}")
print(f"Number of classes: {num_labels}")
print("="*60)

trainer.train()

print("\n" + "="*60)
print("Single-label training complete!")
print("="*60)